### Linear algebra from scratch — Week 1.

Implement every function using only Python builtins and, where a loop would be
pointlessly slow, plain NumPy array indexing. **Do not call the NumPy function
that already does the job.** `np.dot`, `np.linalg.norm`, and friends are what
the tests compare your work against; using them makes the tests tautological.

The point is not that you will ever ship `dot_product` in production. The point
is that when you later read "attention computes a scaled dot product between
queries and keys," you have a physical intuition for what that means rather than
a memorized phrase.

Interview relevance: every embedding question, every similarity question, and
the entire first half of any transformer explanation reduces to this file.

In [1]:
from __future__ import annotations
import numpy as np
from numpy.typing import NDArray

Vector = NDArray[np.float64]
Matrix = NDArray[np.float64]

In [2]:
def check_1D(a: Vector) -> bool:
    if a.ndim == 1:
        return True
    else:
        return False
def check_0V(a: Vector) -> bool:
    for i in range(a.size):
        if a[i] > 0.0:
            return True
    return False

In [3]:
def dot_product(a: Vector, b: Vector) -> float:
    """Return the dot product of two vectors.

    The dot product answers "how much does a point in the direction of b?",
    scaled by both magnitudes. It is the single most important operation in
    machine learning: a linear layer is a batch of dot products, and so is
    attention.

    Args:
        a: 1-D array of shape (n,).
        b: 1-D array of shape (n,).

    Returns:
        The scalar sum of elementwise products.

    Raises:
        ValueError: if the shapes differ or the inputs are not 1-D.
    """
    if not(check_1D(a) and check_1D(b)):
        raise ValueError("Not 1D vector")
    if a.size != b.size:
        raise ValueError ("Vectors length not same")
    dot_product=0.0
    for i in range(a.size):
        dot_product += a[i]*b[i]
    return dot_product  

In [4]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(dot_product(a,b))

32.0


In [5]:
def L2(v :Vector) -> float:
    sqr = 0.0
    for i in range(v.size):
        sqr +=  v[i]*v[i]
    return np.sqrt(sqr)

def L1(v :Vector) -> float:
    norm = 0.0
    for i in range(v.size):
        norm += v[i]
    return norm
        
def Linf(v : Vector) -> float:
    max = v[0]
    for i in range(v.size):
        if max <= v[i]:
            max = v[i]
    return max

def norm(v: Vector, p = 2) -> float:
    """Return the p-norm of a vector.

    p=1 is Manhattan (sum of absolute values), p=2 is Euclidean, p=inf is the
    maximum absolute component. L1 and L2 show up again in Week 5 as the two
    regularization penalties — L1 produces sparsity because its gradient is
    constant, L2 shrinks smoothly because its gradient is proportional to the
    weight. Understand the geometry here and that result stops being a fact you
    memorize.

    Args:
        v: 1-D array.
        p: Order of the norm. Must be >= 1, or float("inf").

    Returns:
        The p-norm as a float.
    """
    if not(check_1D(v)):
        raise ValueError("Not 1D vector")
    match(p):
        case 1:
            print(f"Manhattan Norm is : {L1(v)}")
        case 2:
            print(f"Euclidean Norm is : {L2(v)}")
        case 'inf':
            print(f"Maximum Norm is : {Linf(v)}")
        case _:
            raise ValueError("Not valid Norm")
    

In [6]:
a=np.array([3.0, 4.0])
norm(a, 1)
norm(a, 2)
norm(a, 'inf')

Manhattan Norm is : 7.0
Euclidean Norm is : 5.0
Maximum Norm is : 4.0


In [7]:
def normalize(v: Vector) -> Vector:
    """Return v scaled to unit L2 length.

    Raises:
        ValueError: if v is the zero vector (no direction to preserve).
    """
    if not check_0V(v):
        raise ValueError("v is the zero vector (no direction to preserve)")
    l1 = L2(v)
    norm = v / l1
    return norm

In [8]:
a=np.array([3.0, 4.0])
print(f"Normaliztion of vector{a} : {normalize(a)}")

Normaliztion of vector[3. 4.] : [0.6 0.8]


In [9]:
def cosine_similarity(a: Vector, b: Vector) -> float:
    """Return the cosine of the angle between two vectors.

    This is the similarity metric behind every vector database you will build
    in Month 10. It measures direction only, discarding magnitude — which is
    usually what you want for embeddings, and occasionally exactly wrong.

    Note the relationship to dot product: for unit-length vectors they are
    identical. That is why embedding models normalize their outputs, and why
    pgvector's `<=>` operator is cheap.

    Returns:
        A value in [-1, 1]. 1 means same direction, 0 orthogonal, -1 opposite.

    Raises:
        ValueError: if either vector is the zero vector.
    """
    ab = dot_product(a,b)
    al = L2(a)
    bl = L2(b)
    return (ab/(al*bl))

In [10]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(f"Cosine Similarity of vectors {a} , {b} : {cosine_similarity(a,b)}")

Cosine Similarity of vectors [1. 2. 3.] , [4. 5. 6.] : 0.9746318461970762


In [11]:
def euclidean_distance(a: Vector, b: Vector) -> float:
    """Return the L2 distance between two points."""
    sum = 0.0
    for i in range(a.size):
        sum += (a[i] - b[i]) * (a[i] - b[i])
    return np.sqrt(sum)

In [12]:
a = np.array([1.0, 2.0])
b = np.array([4.0, 5.0])
print(f"Euclidean Distance of vectors {a} , {b} : {euclidean_distance(a,b)}")

Euclidean Distance of vectors [1. 2.] , [4. 5.] : 4.242640687119285


In [13]:
def project(v: Vector, onto: Vector) -> Vector:
    """Return the vector projection of v onto another vector.

    The component of v that lies along `onto`. This is the operation underneath
    least squares (Week 5): fitting a line is projecting the target vector onto
    the column space of the design matrix.

    Raises:
        ValueError: if `onto` is the zero vector.
    """
    if not check_0V(onto):
        raise ValueError("ValueError: if `onto` is the zero vector")
    vonto = dot_product(v, onto)
    ontosq = 0.0
    for i in range(onto.size):
        ontosq += onto[i]*onto[i]
    return (vonto/ontosq)*onto

In [14]:
a = np.array([1.0, 2.0])
b = np.array([4.0, 5.0])
print(f"Project vector {a} onto {b} : {project(a,b)}")

Project vector [1. 2.] onto [4. 5.] : [1.36585366 1.70731707]


In [15]:
def orthogonal_component(v: Vector, onto: Vector) -> Vector:
    """Return the part of v orthogonal to `onto`, i.e. v minus its projection.

    This is the residual. In Week 5 you will discover that least squares
    minimizes exactly this quantity.
    """
    vproj = project(v, onto)
    o_comp = v - vproj
    return o_comp

In [16]:
a = np.array([1.0, 2.0])
b = np.array([4.0, 5.0])
print(f"Orthogonal component vector {a} onto {b} : {orthogonal_component(a,b)}")

Orthogonal component vector [1. 2.] onto [4. 5.] : [-0.36585366  0.29268293]


In [17]:
def angle_between(a: Vector, b: Vector, degrees: bool = False) -> float:
    """Return the angle between two vectors.

    Clamp the cosine into [-1, 1] before calling arccos — floating point will
    hand you 1.0000000000000002 and arccos will hand you back a NaN. This is a
    real bug that ships in real similarity-search code.
    """
    adotb = dot_product(a,b)
    amag  = L2(a)
    bmag  = L2(b)
    cost = max(-1 , min(1 , adotb/(amag * bmag)))
    cosangle = np.arccos(cost)
    if degrees:
        cosangle = np.degrees(cosangle)
    return cosangle

In [18]:
a = np.array([3.0, 4.0])
b = np.array([2.0, 1.0])
print(f"Angle between vector {a} onto {b} : {angle_between(a,b)}")
print(f"Angle between vector {a} onto {b} : {angle_between(a,b, True)}")

Angle between vector [3. 4.] onto [2. 1.] : 0.46364760900080615
Angle between vector [3. 4.] onto [2. 1.] : 26.565051177077994


## Matrix Operations

In [19]:
def matmul(A: Matrix, B: Matrix) -> Matrix:
    """Multiply two matrices.

    Write the triple loop first. Get it right. Then, if you want, write the
    vectorized version and time both — the gap (typically 100-1000x) is a
    lesson about why GPUs matter that is much more convincing when you measure
    it yourself.

    Args:
        A: shape (m, k).
        B: shape (k, n).

    Returns:
        Shape (m, n).

    Raises:
        ValueError: if the inner dimensions do not match.
    """
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    B_rows = 1 if len(B.shape) == 1 else B.shape[0]
    B_cols = B.shape[0] if len(B.shape) == 1 else B.shape[1]  
    #print(f"A_r : {A_rows}, A_c : {A_cols}, B_r : {B_rows}, B_c: {B_cols}")
    if A_cols != B_rows:
        raise ValueError(f"ValueError: if the inner dimensions do not match i.e. A : {A_cols} , B : {B_rows}")
    AB = np.zeros((A_rows, B_cols))
    for i in range(B_cols):
        for j in range(B_rows):
            for k in range(A_rows):
                #print(f"AB[{k}{i}]+=A[{k}{j}]*B[{j}{i}]")
                #print(f"AB[{k}{i}]+={A[k,j]}*{B[j,i]}")
                AB[k,i] += (A[k,j]*B[j,i])
            #print("===")
    return(AB)    

In [20]:
a = np.array([[3.0, 4.0] ,[3.0, 4.0]])
b = np.array([[3.0, 4.0, 5.0] ,[3.0, 4.0, 6.0] ])
ab = matmul(a, b)
print(f"Matrix multipication : \n{a}\n     X  \n{b} \n   = \n{ab}")

Matrix multipication : 
[[3. 4.]
 [3. 4.]]
     X  
[[3. 4. 5.]
 [3. 4. 6.]] 
   = 
[[21. 28. 39.]
 [21. 28. 39.]]


In [21]:
def transpose(A: Matrix) -> Matrix:
    """Return the transpose of A without calling `.T`."""
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    AT = np.zeros((A_cols, A_rows))
    for i in range (A_cols):
        for j in range (A_rows):
            if len(A.shape) == 1:
                AT[i, j] = A[i]
            else:
                AT[i, j] = A[j, i]

    return AT

In [22]:
a = np.array([[3.0, 4.0, 5.0] ,[3.0, 4.0, 6.0] ])
print(f"Transpose of Matrix: \n {a} \n is \n {transpose(a)}")

Transpose of Matrix: 
 [[3. 4. 5.]
 [3. 4. 6.]] 
 is 
 [[3. 3.]
 [4. 4.]
 [5. 6.]]


In [23]:
def identity(n: int) -> Matrix:
    """Return the n x n identity matrix."""
    AI = np.zeros((n, n))
    for i in range (n):
        for j in range(n):
            if i==j:
                AI[i,j]=1
    return AI

In [24]:
print(f"Identity matrix for n {5} is \n{identity(5)}")

Identity matrix for n 5 is 
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


In [25]:
def trace(A: Matrix) -> float:
    """Return the sum of the diagonal of a square matrix.

    Raises:
        ValueError: if A is not square.
    """
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    if A_rows != A_cols:
        raise ValueError("ValueError: if A is not square")
    trace = 0.0
    for i in range(A_rows):
        for j in range(A_cols):
            if i==j:
                trace += A[i,j]
    return trace

In [26]:
a = np.array([[3.0, 4.0, 5.0] ,[3.0, 4.0, 6.0], [3.0, 4.0, 6.0] ])
print(f"Trace of matrix is {trace(a)}")

Trace of matrix is 13.0


In [27]:
def is_orthogonal(A: Matrix, tol: float = 1e-8) -> bool:
    """Return True if A^T A is the identity within tolerance.

    Orthogonal matrices rotate and reflect but never stretch. They preserve
    norms and angles, which is why they are numerically well-behaved and why
    QR and SVD are built out of them.
    """
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    if A_rows != A_cols:
        raise ValueError("ValueError: if A is not square")    
    AT = transpose(A)
    I = matmul (AT, A)
    for i in range (A_rows):
        for j in range (A_cols):
            if i==j:
                if I[i,j] != 1:
                    return False
    return True

In [28]:
a = np.array([[0 ,1],[1, 0]])
print(f"Matrix \n {a} \nis Orghogonal : {is_orthogonal(a)}")

Matrix 
 [[0 1]
 [1 0]] 
is Orghogonal : True


In [29]:
def is_symmetric(A: Matrix, tol: float = 1e-8) -> bool:
    """Return True if A equals its transpose within tolerance.

    Symmetric matrices have real eigenvalues and orthogonal eigenvectors, which
    is why covariance matrices (always symmetric) admit PCA cleanly.
    """
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    if A_rows != A_cols:
        raise ValueError("ValueError: if A is not square")  
    AT = transpose(A)   
    for i in range (A_rows):
        for j in range (A_cols):
            if A[i,j] != A[j,i]:
                return False
    return True

In [30]:
a = np.array([[1,2,3], [2,5,6],[3,6,9]])
print(f"Matrix \n {a} \nis Symmetric : {is_symmetric(a)}")

Matrix 
 [[1 2 3]
 [2 5 6]
 [3 6 9]] 
is Symmetric : True


In [31]:
def matrix_vector_product(A: Matrix, v: Vector) -> Vector:
    """Multiply a matrix by a vector.

    Two readings of this operation, and you should be fluent in both:

    1. Each output element is a dot product of a row of A with v.
    2. The output is a linear combination of A's *columns*, weighted by v.

    Reading 2 is the one that makes "the column space" and "rank" click, and it
    is the one most people never internalize.
    """
    A_rows = 1 if len(A.shape) == 1 else A.shape[0]
    A_cols = A.shape[0] if len(A.shape) == 1 else A.shape[1]
    v_rows = v.shape[0] if len(v.shape) == 1 else 1
    v_cols = 1  
    #print(f"A_r : {A_rows}, A_c : {A_cols}, v_r : {v_rows}, B_c: {v_cols}")
    #return 0
    if A_cols != v_rows:
        raise ValueError(f"ValueError: if the inner dimensions do not match i.e. A : {A_cols} , B : {v_rows}")
    Av = np.zeros(A_rows)
    for i in range(A_rows):
        #print(A[i,])
        #print(v[:,0])
        Av[i] = dot_product(A[i,],v)
    return Av

In [32]:
a = np.array([[1,2,3], [5,7,8],[4,6,9]])
#b = np.array([[1], [3], [4]])
b = np.array([1 , 3, 4 ])
print(f"Matrix Product using vector is :\n {matrix_vector_product(a, b)}")
c = np.array([[1], [3], [4]])
ac = matmul(a, c)
print(f"Matrix multipication : \n{a}\n     X  \n{b} \n   = \n{ac}")

Matrix Product using vector is :
 [19. 58. 58.]
Matrix multipication : 
[[1 2 3]
 [5 7 8]
 [4 6 9]]
     X  
[1 3 4] 
   = 
[[19.]
 [58.]
 [58.]]


In [33]:
def gram_schmidt(vectors: list[Vector]) -> list[Vector]:
    """Return an orthonormal basis for the span of the input vectors.

    Classical Gram-Schmidt: for each vector, subtract its projection onto every
    previously accepted basis vector, then normalize. Drop vectors whose
    residual is numerically zero — those are linearly dependent on the ones
    before them.

    Classical Gram-Schmidt is numerically unstable for near-dependent inputs.
    Implement it anyway, then implement `modified_gram_schmidt` and construct a
    case where they disagree. That exercise is worth more than the function.

    Args:
        vectors: List of 1-D arrays of equal length.

    Returns:
        A list of orthonormal vectors spanning the same subspace. May be
        shorter than the input if the input was linearly dependent.
    """
    GSS = []
    GSSR = []
    tolerance = 1e-12
    for i in range (len(vectors)):
        u = np.zeros(vectors[i][0].size)
        for j in range(i+1):
            #print(f"i : {i}, j : {j}")
            if i != j:
                u +=  (dot_product(GSS[j],vectors[i][0])) * GSS[j]
            else:
                if i !=0:
                    q = vectors[i][0] - u
                    trk = L2(q)
                    if trk < tolerance:
                        GSS.append(np.zeros(vectors[i][0].size))
                    else:
                        GSS.append((1 / trk) * q)
                        GSSR.append(GSS[i])
                else:
                    GSS.append((1 / L2(vectors[j][0])) * vectors[i][0])
                    GSSR.append(GSS[i])
        #print(GSS[i]) 
    GSS = []
    return GSSR   

In [34]:
a = np.array([1, 1, 0])
b = np.array([1, 0, 1])
c = np.array([0, 1, 1])
vecs = [[a], [b], [c]]
abc = gram_schmidt(vecs)
print(abc)

[array([0.70710678, 0.70710678, 0.        ]), array([ 0.40824829, -0.40824829,  0.81649658]), array([-0.57735027,  0.57735027,  0.57735027])]


In [35]:
def modified_gram_schmidt(vectors: list[Vector]) -> list[Vector]:
    """Numerically stable Gram-Schmidt.

    Same output in exact arithmetic; substantially better in floating point.
    The difference: subtract each projection immediately as you go, rather than
    computing all projections against the original vector.

    Stretch goal: build a nearly-dependent input where the classical version
    produces a basis that fails `is_orthogonal` and this one does not.
    """
    pass

In [36]:
def rank(A: Matrix, tol: float = 1e-10) -> int:
    """Return the rank of A: the dimension of its column space.

    Implement via Gram-Schmidt on the columns and count what survives. In Week 2
    you will compute it again from the singular values and find that far more
    robust — that comparison is the lesson.
    """
    raise NotImplementedError("Week 1")

In [37]:
a = np.array([1, 2, 3, 4])
b = np.array([2, 4, 6, 8])
c = np.array([3, 6, 9, 12])
d = np.array([-1, -2, -3, -4])

vecs = [[a], [b], [c], [d]]
print(f"Rank of vector vecs is : {len(gram_schmidt(vecs))}")
a = np.array([1, 0, 0, 0])
b = np.array([0, 1, 0, 0])
c = np.array([1, 1, 0, 0])
d = np.array([2, -1, 0, 0])

vecs = [[a], [b], [c], [d]]
print(f"Rank of vector vecs is : {len(gram_schmidt(vecs))}")
a = np.array([1, 0, 0, 0])
b = np.array([0, 1, 0, 0])
c = np.array([0, 0, 1, 0])
d = np.array([1, 2, 3, 0])

vecs = [[a], [b], [c], [d]]
print(f"Rank of vector vecs is : {len(gram_schmidt(vecs))}")
a = np.array([1, 0, 0, 0])
b = np.array([0, 1, 0, 0])
c = np.array([0, 0, 1, 0])
d = np.array([0, 0, 0, 1])

vecs = [[a], [b], [c], [d]]
print(f"Rank of vector vecs is : {len(gram_schmidt(vecs))}")

Rank of vector vecs is : 1
Rank of vector vecs is : 2
Rank of vector vecs is : 3
Rank of vector vecs is : 4


In [38]:
def batch_cosine_similarity(query: Vector, matrix: Matrix) -> Vector:
    """Cosine similarity between one query vector and every row of a matrix.

    This is vector search. Every retrieval system you build in Months 7-11 is
    this function plus an index that avoids computing all of it.

    Implement it as a single matrix-vector product against pre-normalized rows,
    not as a Python loop over rows. Then time both at n=100_000 and write the
    numbers in your week check-in.

    Args:
        query: shape (d,).
        matrix: shape (n, d).

    Returns:
        Shape (n,), similarity to each row.
            mq = matrix_vector_product(matrix, query)
            
    query_norm = np.linalg.norm(query)    
    matrix_norms = np.linalg.norm(matrix, axis=1)
    return ((matrix @ query) / (query_norm* matrix_norms ))
    
    """
    mq = matrix_vector_product(matrix, query)
    queryL2 = L2(query)
    m_norm = np.zeros(query.size)
    for i in range (matrix.shape[0]):
        m_norm[i]=L2(matrix[i,:])
    return((mq) / (queryL2 * m_norm ))

In [39]:
a = np.array([1.0 , 3.0 , 2.0])
b = np.array([[1,2,3], [5,7,8],[4,6,9]])
batch_cosine_similarity(a,b)

array([0.92857143, 0.95553309, 0.92697955])

In [40]:
def top_k_similar(query: Vector, matrix: Matrix, k: int = 5) -> tuple[NDArray, Vector]:
    """Return the indices and scores of the k most similar rows.

    Use `np.argpartition` rather than a full sort: you need the top k, not a
    total ordering, and the difference is O(n) versus O(n log n). At n = 10
    million this is the difference between a snappy search and a timeout.

    Returns:
        (indices, scores), both length k, sorted by descending score.
    """
    similarity = batch_cosine_similarity(query, matrix)
    idx = np.argpartition(similarity, k)[k:]
    idx = idx [::-1]
    return [idx ,  similarity[idx]]

In [41]:
v = np.array([14, 2, 3, 4, 5, 6, 7, 8, 9, 10])
A = np.array([
    [1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
    [81, 82, 83, 84, 85, 86, 87, 88, 89, 90],
    [21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
    [51, 52, 53, 54, 55, 56, 57, 58, 59, 60],    
    [31, 32, 33, 34, 35, 36, 37, 38, 39, 40],
    [41, 42, 43, 44, 45, 46, 47, 48, 49, 50],
    [61, 62, 63, 64, 65, 66, 67, 68, 69, 70],
    [71, 72, 73, 74, 75, 76, 77, 78, 79, 80],
    [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
])
top_k_similar(v, A)

[array([5, 1, 3, 6, 4]),
 array([0.8988236 , 0.89792811, 0.89955347, 0.89802226, 0.89736094])]